# 02: Data Cleaning

In this notebook, you'll learn to identify and fix common data quality issues using Pandas.

## Learning Objectives

- Identify missing values using `isnull()` and `sum()`
- Handle missing data with `dropna()`, `fillna()`, and interpolation
- Remove duplicate rows
- Fix data types and convert columns
- Clean string data (whitespace, casing, patterns)
- Work with a synthetic dirty dataset
- Compare before/after states

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

## 2. Create a Dirty Dataset

Real-world data is messy. Let's simulate that with a synthetic dataset that has common problems.

In [ ]:
# Synthetic dirty dataset: customer orders
dirty_data = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "customer_name": ["Alice Smith", " bob jones", "CHARLIE BROWN", "Diana Prince", 
                      " Eve Torres", "Frank Castle", "grace hopper", "", None, "Alice Smith"],
    "email": ["alice@mail.com", "bob@mail.com", "charlie@@mail.com", "diana@mail.com",
              "eve@mail.com", "frank@mail.com", None, "grace@mail.com", "henry@mail.com", "alice@mail.com"],
    "age": [28, -5, 42, 31, 200, 45, 33, 29, None, 28],
    "salary": [75000, 82000, None, 68000, 72000, 110000, 88000, 71000, 79000, 75000],
    "department": ["Engineering", "Marketing", "engineering", "HR", 
                   "Marketing", "Engineering", "HR", "MARKETING", None, "Engineering"],
    "join_date": ["2021-03-15", "2020/07/01", "Jan 20, 2019", "2022-11-05",
                   "2023-02-14", "15-06-2018", "2021-09-12", "2022-04-22", "2023-01-01", "2021-03-15"],
    "phone": ["555-1234", "555 5678", "(555)9012", "555-3456",
              "5557890", "555-0123", "555 4567", "555-8901", "555-2345", "555-1234"]
})

print("Dirty dataset created!")
print(f"Shape: {dirty_data.shape}")
dirty_data

**Problems in this dataset:**
- Missing values (None, empty strings)
- Duplicates (row 0 and row 9)
- Inconsistent casing ("CHARLIE BROWN", "grace hopper")
- Leading/trailing whitespace (" bob jones", " Eve Torres")
- Invalid values (negative age, age=200)
- Inconsistent date formats ("2020/07/01", "Jan 20, 2019", "15-06-2018")
- Inconsistent phone number formats
- Inconsistent department names ("engineering" vs "Engineering")

## 3. Identifying Missing Values

In [ ]:
# Check for null values
print("=== Null values per column ===")
print(dirty_data.isnull().sum())

In [ ]:
# Total null values
total_nulls = dirty_data.isnull().sum().sum()
print(f"Total null values: {total_nulls}")

# Percentage of missing data
print("\n=== Missing % per column ===")
missing_pct = (dirty_data.isnull().sum() / len(dirty_data) * 100).round(1)
print(missing_pct[missing_pct > 0])

In [ ]:
# Also detect empty strings as missing
print("=== Empty strings per column ===")
for col in dirty_data.columns:
    empty_count = (dirty_data[col].astype(str).str.strip() == "").sum()
    if empty_count > 0:
        print(f"{col}: {empty_count}")

## 4. Handling Missing Data

### 4.1 Drop Rows with Missing Values

In [ ]:
# Drop rows where customer_name is missing
df_dropped = dirty_data.dropna(subset=["customer_name"])
print(f"Before: {len(dirty_data)} rows")
print(f"After:  {len(df_dropped)} rows")
df_dropped

### 4.2 Fill with a Constant

In [ ]:
# Fill missing email with "unknown@mail.com"
df_filled = dirty_data.copy()
df_filled["email"] = df_filled["email"].fillna("unknown@mail.com")
print("Filled missing emails with 'unknown@mail.com'")
df_filled[["customer_name", "email"]]

### 4.3 Fill with Statistical Values (Mean/Median/Mode)

In [ ]:
# Fill missing salary with median
df_filled = dirty_data.copy()
median_salary = df_filled["salary"].median()
df_filled["salary"] = df_filled["salary"].fillna(median_salary)
print(f"Filled missing salary with median: ${median_salary:,.0f}")

# Fill missing department with mode
mode_dept = df_filled["department"].mode()[0]
df_filled["department"] = df_filled["department"].fillna(mode_dept)
print(f"Filled missing department with mode: {mode_dept}")

### 4.4 Interpolation (for numeric sequences)

In [ ]:
# Interpolation fills values based on surrounding data
df_interp = dirty_data.copy()
df_interp["salary"] = df_interp["salary"].interpolate()
print("Interpolated missing salary values:")
df_interp[["order_id", "salary"]]

## 5. Removing Duplicates

In [ ]:
# Find duplicates
print(f"Duplicate rows: {dirty_data.duplicated().sum()}")

# Show duplicate rows
dirty_data[dirty_data.duplicated(keep=False)]

In [ ]:
# Remove duplicates (keep first occurrence)
df_no_dups = dirty_data.drop_duplicates()
print(f"Before: {len(dirty_data)} rows")
print(f"After:  {len(df_no_dups)} rows")

# Remove duplicates based on specific columns
df_no_dups_email = dirty_data.drop_duplicates(subset=["email"], keep="first")
print(f"\nAfter removing duplicate emails: {len(df_no_dups_email)} rows")

## 6. Fixing Data Types

In [ ]:
df_typed = dirty_data.copy()

# Convert age to numeric (coerce invalid values to NaN)
df_typed["age"] = pd.to_numeric(df_typed["age"], errors="coerce")
print("Age after numeric conversion:")
print(df_typed["age"])

# Check for invalid ages after conversion
print(f"\nInvalid ages (NaN): {df_typed['age'].isnull().sum()}")

In [ ]:
# Handle invalid age values
# Cap age at reasonable range (0-120)
df_typed.loc[df_typed["age"] < 0, "age"] = np.nan
df_typed.loc[df_typed["age"] > 120, "age"] = np.nan

print("Age after fixing invalid values:")
print(df_typed["age"])
print(f"NaN count: {df_typed['age'].isnull().sum()}")

In [ ]:
# Convert department to categorical
df_typed["department"] = df_typed["department"].astype("category")
print(f"Department dtype: {df_typed['department'].dtype}")

## 7. String Cleaning

In [ ]:
df_str = dirty_data.copy()

# Strip whitespace from customer names
df_str["customer_name"] = df_str["customer_name"].str.strip()
print("After stripping whitespace:")
print(df_str["customer_name"].tolist())

In [ ]:
# Title case for names
df_str["customer_name"] = df_str["customer_name"].str.title()
print("After title case:")
print(df_str["customer_name"].tolist())

In [ ]:
# Standardize department names (title case)
df_str["department"] = df_str["department"].str.strip().str.title()
print("Departments after standardization:")
print(df_str["department"].value_counts())

In [ ]:
# Clean email: lowercase and remove double @
df_str["email"] = df_str["email"].str.lower().str.replace("@@", "@", regex=False)
print("Emails after cleanup:")
print(df_str["email"].tolist())

In [ ]:
# Standardize phone numbers: remove all non-digits
df_str["phone"] = df_str["phone"].str.replace(r"\D", "", regex=True)
print("Phone numbers after cleanup:")
print(df_str["phone"].tolist())

## 8. Complete Cleaning Pipeline

Let's combine everything into one cleaning function.

In [ ]:
def clean_customer_data(df):
    """Clean the customer order dataset."""
    df = df.copy()
    
    # --- Remove duplicates ---
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {before - len(df)} duplicate rows")
    
    # --- String cleanup ---
    df["customer_name"] = df["customer_name"].str.strip().str.title()
    df["email"] = df["email"].str.strip().str.lower().str.replace("@@", "@", regex=False)
    df["department"] = df["department"].str.strip().str.title()
    df["phone"] = df["phone"].str.replace(r"\D", "", regex=True)
    
    # --- Handle missing values ---
    # Drop rows with missing critical fields
    df = df.dropna(subset=["customer_name"])
    
    # Drop empty customer names
    df = df[df["customer_name"] != ""]
    
    # Fill missing email
    df["email"] = df["email"].fillna("unknown@mail.com")
    
    # Fill missing department with mode
    df["department"] = df["department"].fillna(df["department"].mode()[0])
    
    # --- Fix data types ---
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df.loc[df["age"] < 0, "age"] = np.nan
    df.loc[df["age"] > 120, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())
    df["age"] = df["age"].astype(int)
    
    # Fill missing salary with median
    df["salary"] = df["salary"].fillna(df["salary"].median())
    
    # --- Reset index ---
    df = df.reset_index(drop=True)
    
    print(f"Cleaned: {len(df)} rows remaining")
    return df

In [ ]:
# Apply the cleaning pipeline
clean_df = clean_customer_data(dirty_data)

print("\n=== Cleaned Dataset ===")
clean_df

## 9. Before/After Comparison

In [ ]:
# Compare missing values
print("=== Missing Values: Before ===")
print(dirty_data.isnull().sum())
print(f"\nTotal: {dirty_data.isnull().sum().sum()}")

print("\n=== Missing Values: After ===")
print(clean_df.isnull().sum())
print(f"Total: {clean_df.isnull().sum().sum()}")

In [ ]:
# Compare row counts
print(f"Before: {len(dirty_data)} rows")
print(f"After:  {len(clean_df)} rows")
print(f"Removed: {len(dirty_data) - len(clean_df)} rows")

In [ ]:
# Compare data types
print("=== Data Types: Before ===")
print(dirty_data.dtypes)

print("\n=== Data Types: After ===")
print(clean_df.dtypes)

## 10. Save Cleaned Data

In [ ]:
import os

os.makedirs("data", exist_ok=True)
clean_df.to_csv("data/customers_cleaned.csv", index=False)
print("Saved to data/customers_cleaned.csv")

## Exercises

### Exercise 1: Create and clean your own dirty dataset

Create a dirty dataset with at least 10 rows and these issues:
- 2 missing values in different columns
- 1 duplicate row
- 2 inconsistent string values
- 1 invalid numeric value

Then write a cleaning function that fixes all issues.

### Exercise 2: Handle a different dirty dataset

Use this dataset:

```python
dirty_products = pd.DataFrame({
    "product": ["Widget A", " widget b ", "WIDGET C", None, "Widget D", "widget a"],
    "price": ["$10.00", "$20.50", "15.00", "$25.00", "N/A", "$10.00"],
    "quantity": [100, -5, 200, 150, 75, 100]
})
```

Tasks:
1. Clean the product names (strip, title case)
2. Convert prices to numeric (remove $, handle N/A)
3. Fix invalid quantities
4. Remove duplicates

### Exercise 3: Which strategy for which column?

For each scenario, choose the best missing value strategy and justify:
1. A column with 5% missing values in a 1M row dataset
2. A column with 90% missing values
3. A time-series column with a few gaps
4. A categorical column where mode is 80% of values

In [ ]:
# Your code here for Exercise 1
# ...